In [44]:
# Setup colab
from google.colab import drive
drive.mount('/content/drive')
print("✅ Drive mounted")
import os
os.chdir('/content/drive/MyDrive/reranking_rag_and_qw/notebook')
print(f"✅ Working directory: {os.getcwd()}")
print("Contents:", os.listdir("."))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted
✅ Working directory: /content/drive/.shortcut-targets-by-id/1nJ6qfsYQS_sTYFLUl7Y2q0Uv9A3cadKk/reranking_rag_and_qw/notebook
Contents: ['.gitkeep', '01_setup_baseline.ipynb', '02_query_writing.ipynb', '03_reranking.ipynb', '04_train.ipynb']


In [ ]:
# Install requirements
%pip install -r ../requirements.txt
%pip install -q bitsandbytes accelerate

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 115.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.7/625.7 kB 52.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.4/85.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━

In [45]:
%pip install -q "sentence-transformers>=3.0"
%pip install -q "jsonlines"
%pip install -q "rank_bm25"
%pip install -q "rouge_score"
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# 4. Train Custom Reranker
Huấn luyện reranker custom trên dữ liệu Vietnamese

## 4.1 Chuẩn bị dữ liệu training

In [46]:
%pip install -q "sentence-transformers>=3.0" datasets scikit-learn jsonlines

In [47]:
import sys, os
root_path = os.path.abspath(os.path.join('..'))
if root_path not in sys.path:
    sys.path.append(root_path)

import json
import random
import jsonlines
import numpy as np
from pathlib import Path
from tqdm import tqdm

from src.data.data_loader import DataLoader

data_loader = DataLoader("../data")
print("✅ Imports successful")

✅ Imports successful


In [48]:
# Gom dữ liệu từ tất cả datasets
DATASETS = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]

all_train_data = []
for ds in DATASETS:
    try:
        data = data_loader.load_dataset(ds, "train")
        all_train_data.extend(data)
        print(f"✅ {ds}: {len(data)} samples")
    except Exception as e:
        print(f"⚠️  {ds}: {e}")

print(f"\n📊 Total: {len(all_train_data)} samples")

✅ vhealthqa: 7009 samples
✅ uit_viquad2: 28454 samples
✅ vietnamese_rag: 6728 samples
✅ vimpa: 8041 samples

📊 Total: 50232 samples


In [49]:
def extract_fields(sample: dict):
    """Trích xuất query và context từ sample, bỏ qua context là URL."""
    # Ưu tiên field 'query' trước, sau đó 'question'
    query = sample.get("query") or sample.get("question") or ""
    context = sample.get("context") or sample.get("passage") or sample.get("text") or ""

    # Bỏ qua nếu context là URL
    if context.startswith("http"):
        context = sample.get("ground_truth") or sample.get("answer") or ""

    return query.strip(), context.strip()

# Tạo positive pairs
positive_pairs = []
for s in all_train_data:
    q, ctx = extract_fields(s)
    if q and ctx and not ctx.startswith("http"):
        positive_pairs.append({"query": q, "document": ctx, "label": 1})

print(f"✅ {len(positive_pairs)} positive pairs")
print("\nVí dụ:")
print(json.dumps(positive_pairs[0], ensure_ascii=False, indent=2))

✅ 50232 positive pairs

Ví dụ:
{
  "query": "Đang chích ngừa viêm gan B có chích ngừa Covid-19 được không?",
  "document": "Nếu anh/chị đang tiêm ngừa vaccine phòng bệnh viêm gan B, anh/chị vẫn có thể tiêm phòng vaccine phòng Covid-19, tuy nhiên vaccine Covid-19 phải được tiêm cách trước và sau mũi vaccine viêm gan B tối thiểu là 14 ngày.",
  "label": 1
}


In [50]:
# Tạo negative pairs (random hard negatives)
random.seed(42)
all_docs = [p["document"] for p in positive_pairs]

negative_pairs = []
sample_pool = positive_pairs[:len(positive_pairs) // 2]

for pp in sample_pool:
    neg_doc = random.choice(all_docs)
    while neg_doc == pp["document"]:
        neg_doc = random.choice(all_docs)
    negative_pairs.append({"query": pp["query"], "document": neg_doc, "label": 0})

# Gộp và xáo trộn
all_pairs = positive_pairs + negative_pairs
random.shuffle(all_pairs)

print(f"✅ Positive: {len(positive_pairs)}  |  Negative: {len(negative_pairs)}")
print(f"📊 Total pairs: {len(all_pairs)}")

✅ Positive: 50232  |  Negative: 25116
📊 Total pairs: 75348


In [8]:
# Lưu ra file jsonl
train_file = Path("../data/reranker_training_data.jsonl")
train_file.parent.mkdir(parents=True, exist_ok=True)

with jsonlines.open(train_file, mode="w") as writer:
    writer.write_all(all_pairs)

print(f"✅ Saved: {train_file}  ({train_file.stat().st_size / 1024:.0f} KB)")

✅ Saved: ../data/reranker_training_data.jsonl  (251876 KB)


## 4.2 Train Cross-Encoder với CrossEncoderTrainer (sentence-transformers ≥ 3.x)

In [51]:
import torch
import gc
import os

# Buộc Python dọn dẹp rác
gc.collect()

# Giải phóng bộ nhớ đệm của CUDA
torch.cuda.empty_cache()

# Thiết lập chống phân mảnh
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

print(f"GPU RAM hiện tại: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")

GPU RAM hiện tại: 492.17 MB


In [52]:
import torch
from datasets import Dataset
from sentence_transformers import CrossEncoder
from sentence_transformers.cross_encoder import CrossEncoderTrainer, CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator
from sklearn.model_selection import train_test_split

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BASE_MODEL = "BAAI/bge-reranker-base"
OUTPUT_DIR = Path("../models/bge-reranker-vi")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device : {DEVICE}")
print(f"Model  : {BASE_MODEL}")

Device : cuda
Model  : BAAI/bge-reranker-base


In [53]:
# Load dữ liệu và tách train/val
with jsonlines.open('../data/reranker_training_data.jsonl') as reader:
    pairs = list(reader)

queries   = [p["query"]    for p in pairs]
documents = [p["document"] for p in pairs]
labels    = [float(p["label"]) for p in pairs]

idx = list(range(len(labels)))
train_idx, val_idx = train_test_split(idx, test_size=0.15, random_state=42)

def make_hf_dataset(indices):
    return Dataset.from_dict({
        "sentence1": [queries[i]   for i in indices],
        "sentence2": [documents[i] for i in indices],
        "label":     [labels[i]    for i in indices],
    })

train_dataset = make_hf_dataset(train_idx)
val_dataset   = make_hf_dataset(val_idx)

print(f"Train: {len(train_dataset)}  |  Val: {len(val_dataset)}")

Train: 64045  |  Val: 11303


In [54]:
# Khởi tạo model
model = CrossEncoder(
    BASE_MODEL,
    num_labels=1,
    device=DEVICE,
    trust_remote_code=True,
)
print("✅ Model loaded")

config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Model loaded


In [ ]:
# Evaluator dùng sigmoid (không dùng argmax vì output là logit đơn)
evaluator = CEBinaryClassificationEvaluator(
    sentence_pairs=list(zip(val_dataset["sentence1"], val_dataset["sentence2"])),
    labels=val_dataset["label"],
    name="val",
)

# Training arguments
training_args = CrossEncoderTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=16,
    per_device_eval_batch_size=4,
    warmup_ratio=0.1,
    learning_rate=2e-5,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_accuracy",
    logging_steps=50,
    fp16=True,
    report_to="none",
    optim="adamw_torch_fused",
)

print("✅ Training args ready")

/tmp/ipykernel_1072/1599936091.py:2: DeprecationWarning: This evaluator has been deprecated in favor of the more general CrossEncoderClassificationEvaluator. Please use CrossEncoderClassificationEvaluator instead, which supports both binary and multi-class evaluation. It accepts approximately the same inputs as this evaluator.
  evaluator = CEBinaryClassificationEvaluator(


✅ Training args ready


In [ ]:
import gc
import torch

del train_dataset, val_dataset
gc.collect()
torch.cuda.empty_cache()

trainer = CrossEncoderTrainer(
    model=model,
    args=training_args,
    train_dataset=make_hf_dataset(train_idx),
    eval_dataset=make_hf_dataset(val_idx),
    evaluator=evaluator,
)

trainer.train()
print(f"\n✅ Training complete → {OUTPUT_DIR}")

Token indices sequence length is longer than the specified maximum sequence length for this model (742 > 512). Running this sequence through the model will result in indexing errors


Epoch,Training Loss,Validation Loss,Val Accuracy,Val Accuracy Threshold,Val F1,Val F1 Threshold,Val Precision,Val Recall,Val Average Precision
1,0.032924,0.026730,0.991595,0.516884,0.993631,0.516884,0.991968,0.995299,0.999707


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
import numpy as np
from collections import defaultdict

def evaluate_ranking(model, dataset, k=10):
    queries = dataset["sentence1"]
    docs = dataset["sentence2"]
    labels = dataset["label"]

    # Gom theo query
    group = defaultdict(list)
    for q, d, l in zip(queries, docs, labels):
        group[q].append((d, l))

    recall_at_k = []
    mrr = []

    for q, items in group.items():
        pairs = [(q, d) for d, _ in items]
        scores = model.predict(pairs)

        # sort theo score
        ranked = sorted(zip(items, scores), key=lambda x: x[1], reverse=True)

        # labels sorted
        sorted_labels = [l for ((_, l), _) in ranked]

        # --- Recall@k ---
        top_k = sorted_labels[:k]
        total_relevant = sum(l for _, l in items)

        if total_relevant > 0:
            recall = sum(top_k) / total_relevant
            recall_at_k.append(recall)

        # --- MRR ---
        rr = 0
        for i, l in enumerate(sorted_labels):
            if l == 1:
                rr = 1 / (i + 1)
                break
        mrr.append(rr)

    return {
        "Recall@{}".format(k): np.mean(recall_at_k),
        "MRR": np.mean(mrr),
    }

In [ ]:
# classification metrics
evaluator(model)

# ranking metrics
ranking_metrics = evaluate_ranking(model, val_dataset, k=5)

print("\n📊 Ranking Metrics:")
for k, v in ranking_metrics.items():
    print(f"{k}: {v:.4f}")

In [ ]:
# Lưu model cuối
model.save(str(OUTPUT_DIR / "final"))
print(f"✅ Model saved to {OUTPUT_DIR / 'final'}")

## 4.3 Đánh giá: Fine-tuned vs Pre-trained

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(ce_model, s1_list, s2_list, true_labels):
    """Sigmoid thresholds at 0.5 để ra nhãn nhị phân."""
    raw = ce_model.predict(list(zip(s1_list, s2_list)))
    # Single logit → sigmoid
    probs = 1.0 / (1.0 + np.exp(-raw))
    preds = (probs >= 0.5).astype(int)
    true  = np.array(true_labels, dtype=int)
    return {
        "accuracy" : accuracy_score(true, preds),
        "precision": precision_score(true, preds, zero_division=0),
        "recall"   : recall_score(true, preds, zero_division=0),
        "f1"       : f1_score(true, preds, zero_division=0),
    }

val_s1  = val_dataset["sentence1"]
val_s2  = val_dataset["sentence2"]
val_lbl = [int(l) for l in val_dataset["label"]]

# Fine-tuned
ft_metrics = evaluate_model(model, val_s1, val_s2, val_lbl)

# Pre-trained baseline
pretrained = CrossEncoder(BASE_MODEL, num_labels=1, device=DEVICE, trust_remote_code=True)
pt_metrics = evaluate_model(pretrained, val_s1, val_s2, val_lbl)

print("\n📊 Results on validation set")
print(f"{'Metric':<12} {'Pre-trained':>12} {'Fine-tuned':>12} {'Δ':>8}")
print("-" * 48)
for k in ft_metrics:
    pt = pt_metrics[k]
    ft = ft_metrics[k]
    print(f"{k:<12} {pt:>12.4f} {ft:>12.4f} {ft-pt:>+8.4f}")

## 4.4 Test trên câu hỏi thực tế

In [ ]:
%pip install faiss-cpu

In [ ]:
from src.retrieval.hybrid_retriever import HybridRetriever
from collections import defaultdict
import numpy as np

DATASETS_TO_EVAL = ["vhealthqa", "uit_viquad2", "vietnamese_rag", "vimpa"]
eval_results = {}

for ds in DATASETS_TO_EVAL:
    try:
        test_samples = data_loader.load_dataset(ds, "test")[:100]
        if not test_samples:
            continue
            
        train_samples = data_loader.load_dataset(ds, "train")
        docs_pool = []
        for s in train_samples:
            _, ctx = extract_fields(s)
            if ctx and not ctx.startswith("http"):
                docs_pool.append(ctx)
                
        retriever = HybridRetriever()
        retriever.build_index(list(set(docs_pool)))
        
        pt_mrr, ft_mrr = [], []
        pt_recall, ft_recall = [], []
        
        for sample in test_samples:
            q, true_ctx = extract_fields(sample)
            if not q or not true_ctx:
                continue
                
            raw_docs = retriever.retrieve([q], top_k=20)
            if not raw_docs:
                continue
                
            pairs = [[q, d["text"]] for d in raw_docs]
            
            pt_scores = pretrained.predict(pairs)
            pt_probs = 1.0 / (1.0 + np.exp(-pt_scores))
            pt_ranked = [doc for _, doc in sorted(zip(pt_probs, raw_docs), key=lambda x: x[0], reverse=True)]
            
            ft_scores = model.predict(pairs)
            ft_probs = 1.0 / (1.0 + np.exp(-ft_scores))
            ft_ranked = [doc for _, doc in sorted(zip(ft_probs, raw_docs), key=lambda x: x[0], reverse=True)]
            
            def calculate_metrics(ranked_docs, truth_text, k=5):
                top_k = [d["text"] for d in ranked_docs[:k]]
                is_hit = any(truth_text in t or t in truth_text for t in top_k)
                recall_val = 1.0 if is_hit else 0.0
                mrr_val = 0.0
                for rank, d in enumerate(ranked_docs):
                    if truth_text in d["text"] or d["text"] in truth_text:
                        mrr_val = 1.0 / (rank + 1)
                        break
                return recall_val, mrr_val

            r_pt, m_pt = calculate_metrics(pt_ranked, true_ctx, k=5)
            r_ft, m_ft = calculate_metrics(ft_ranked, true_ctx, k=5)
            
            pt_recall.append(r_pt)
            pt_mrr.append(m_pt)
            ft_recall.append(r_ft)
            ft_mrr.append(m_ft)
            
        if pt_mrr:
            eval_results[ds] = {
                "PT_Rec@5": np.mean(pt_recall),
                "FT_Rec@5": np.mean(ft_recall),
                "PT_MRR": np.mean(pt_mrr),
                "FT_MRR": np.mean(ft_mrr)
            }
    except Exception:
        pass

print(f"{'Dataset':<16} | {'PT Rec@5':<10} | {'FT Rec@5':<10} | {'PT MRR':<10} | {'FT MRR':<10}")
print("-" * 65)
for ds, metrics in eval_results.items():
    print(f"{ds:<16} | {metrics['PT_Rec@5']:<10.4f} | {metrics['FT_Rec@5']:<10.4f} | {metrics['PT_MRR']:<10.4f} | {metrics['FT_MRR']:<10.4f}")

In [ ]:
TEST_QUERY = "Bệnh đái tháo đường có triệu chứng gì?"
raw_docs = retriever.retrieve([TEST_QUERY], top_k=10)

pairs = [(TEST_QUERY, d["text"]) for d in raw_docs]

def rank_docs(ce_model, query_doc_pairs, doc_list):
    raw = ce_model.predict(query_doc_pairs)
    probs = 1.0 / (1.0 + np.exp(-raw))
    order = np.argsort(probs)[::-1]
    return [(doc_list[i], float(probs[i])) for i in order]

print(f"\nQuery: {TEST_QUERY}\n")
for label, model_obj in [("Pre-trained", pretrained), ("Fine-tuned", model)]:
    ranked = rank_docs(model_obj, pairs, raw_docs)
    print(f"--- {label} ---")
    for i, (doc, score) in enumerate(ranked[:5], 1):
        print(f"  {i}. [{score:.3f}] {doc['text'][:90]}...")
    print()

## 4.5 Lưu model path ra JSON để pipeline dùng lại

In [ ]:
import json as _json

model_info = {
    "cross_encoder_model": str((OUTPUT_DIR / "final").resolve()),
    "base_model": BASE_MODEL,
    "num_labels": 1,
    "score_method": "sigmoid",   # reminder: use sigmoid, NOT argmax
    "val_f1": round(ft_metrics["f1"], 4),
    "val_accuracy": round(ft_metrics["accuracy"], 4),
}

info_path = Path("../models/model_info.json")
info_path.parent.mkdir(parents=True, exist_ok=True)
with open(info_path, "w", encoding="utf-8") as f:
    _json.dump(model_info, f, ensure_ascii=False, indent=2)

print("✅ Model info saved:")
print(_json.dumps(model_info, ensure_ascii=False, indent=2))

In [ ]:
# Load model
# import json
# from sentence_transformers import CrossEncoder
#
# info = json.load(open("../models/model_info.json"))
# model = CrossEncoder(info["cross_encoder_model"], num_labels=info["num_labels"])
# # Sau đó dùng RankReranker(method="cross-encoder", cross_encoder_model=info["cross_encoder_model"])
print("Xem comment ở trên để load lại model trong notebook khác.")